# G1 — GPU speed test: pandas (CPU) vs RAPIDS cuDF (GPU) on raw FDA FAERS files
**Owner:** Karishma  |  
**Project:** DATA230 Group 4  |  
**Run in:** Google Colab with a **GPU runtime**

> This notebook uses the **raw FDA FAERS quarterly files**. It re-creates the *kind* of EDA work our project does — load, missing-rate scan, GLP-1 text matching, group-by, joins, de-duplication, % serious by reporter — once on the CPU with **pandas** and once on the GPU with **RAPIDS cuDF**, and compares them.

The notebook measures all three:
1. **Performance** – seconds per operation, CPU vs GPU, and the speed-up.
2. **Scalability** – the same pipeline on 1×, 2×, 4× the data.
3. **Insight differences** – do both libraries return *identical* answers?

## What RAPIDS / cuDF
RAPIDS is NVIDIA's open-source suite of GPU data-science libraries; **cuDF** is its DataFrame library with a pandas-like API, so `df.groupby(...)`, `df.merge(...)` and `df.str.contains(...)` run on thousands of GPU cores instead of a few CPU cores. It helps most on **large** tables and on **joins, group-bys, sorts and string operations**; on tiny tables the cost of moving data to the GPU can cancel the gain — the scalability test shows where that crossover is.

## 0 · Switch Colab to a GPU runtime (do this first!)
**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save.**

In [ ]:
import os, sys, subprocess, shutil, time, gc, statistics, zipfile, textwrap
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
INSTALL_RAPIDS = False        # set True only if `import cudf` fails below

print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout or "❌ nvidia-smi not found -> no GPU attached")

if INSTALL_RAPIDS:
    # NVIDIA's official Colab installer (see docs.rapids.ai -> Deployment -> Google Colab)
    cmds = ["git clone https://github.com/rapidsai-community/rapidsai-csp-utils.git /content/rapidsai-csp-utils",
            "python /content/rapidsai-csp-utils/colab/pip-install.py"]
    for cmd in cmds:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print((r.stdout + r.stderr)[-1500:])
    print("➡️  Now: Runtime -> Restart session, then run again with INSTALL_RAPIDS = False")

try:
    import cudf
    HAS_GPU = True
    print("✅ cuDF", cudf.__version__)
except Exception as e:
    cudf, HAS_GPU = None, False
    print("⚠️  cuDF not available ->", repr(e))
    print("    The CPU part still runs; GPU columns stay empty. Fix: Runtime -> Change runtime type -> T4 GPU (see above).")

## 1 · Settings
* `QUARTERS` – which FAERS quarters to download (named like the team's `src/ingest.py`: `2025Q1`). **One quarter is enough** (the hand-off counts 1.6 million FDA reports for all of 2025, so one quarter is roughly a quarter of that, with several rows in DRUG per report); two quarters make the CPU run noticeably slower.
* `SCALES` – data multipliers for the scalability test. If Colab says *"Your session crashed (out of memory)"*, use `[1, 2]`.
* `REPEATS` – each operation is timed this many times; the **median** is reported.


In [ ]:
QUARTERS = ["2025Q1"]                    # e.g. ["2025Q1", "2025Q2"]
SCALES   = [1, 2, 4]                     # data multipliers for the scalability test
REPEATS  = 3

REPO_URL = "https://github.com/BrianJBee/data230-group4-project.git"
WORK = Path("/content/faers_raw") if IN_COLAB else Path("./faers_raw")   # raw data stays OUT of the git repo
WORK.mkdir(parents=True, exist_ok=True)

# ---- shared team rules: clone the repo (Colab) or use the local checkout -------------------------
if IN_COLAB:
    REPO_DIR = Path("/content/data230-group4-project")
    if not REPO_DIR.exists():
        r = subprocess.run(f"git clone {REPO_URL} {REPO_DIR}", shell=True, capture_output=True, text=True); print((r.stdout + r.stderr).strip())
    r = subprocess.run("git checkout main && git pull", shell=True, cwd=REPO_DIR, capture_output=True, text=True); print((r.stdout + r.stderr).strip())
    os.chdir(REPO_DIR / "notebooks")
else:
    REPO_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_DIR))

import yaml
from src.definitions import GLP1_KEYWORDS, EXCLUDE_KEYWORDS, SERIOUS_CODES, REPORTER_TYPES
CFG = yaml.safe_load((REPO_DIR / "config.yml").read_text())
DELIMITER, ENCODING = CFG["ingestion"]["delimiter"], CFG["ingestion"]["encoding"]
FAERS_BASE, FAERS_EXT = CFG["download"]["base_url"], CFG["download"]["extension"]

GLP1_REGEX = "|".join(GLP1_KEYWORDS)          # upper-case keywords from definitions.py (drugs + brands)
EXCLUDE_REGEX = "|".join(EXCLUDE_KEYWORDS)    # insulin combination products are left out, as in the cleaning notebook
print("GLP-1 keywords:", len(GLP1_KEYWORDS), "| excluded:", EXCLUDE_KEYWORDS, "| serious codes:", SERIOUS_CODES, "| delimiter", repr(DELIMITER), ENCODING)

# columns we need from each FAERS table (names are matched case-insensitively)
NEEDED = {
    "demo": ["primaryid", "caseid", "caseversion", "age", "age_cod", "sex", "occp_cod", "reporter_country"],
    "drug": ["primaryid", "drug_seq", "role_cod", "drugname", "prod_ai"],
    "outc": ["primaryid", "outc_cod"],
}

## 2 · Get the raw FAERS files
FAERS = the FDA Adverse Event Reporting System. Each quarter is a ZIP with `$`-separated text files: **DEMO** (one row per report), **DRUG** (one row per drug in a report), **OUTC** (outcomes such as death, hospitalisation), plus REAC, INDI, THER, RPSR we do not need here.


In [ ]:
def fetch_quarter(q):
    outdir = WORK / q
    zpath = WORK / f"faers_ascii_{q}.zip"
    if not (outdir.exists() and any(outdir.rglob("*.txt")) or any(outdir.rglob("*.TXT"))):
        if not zpath.exists():
            import requests
            last = None
            for name in (q, q.lower()):                       # the FDA has used both 2025Q1 and 2025q1 over the years
                url = f"{FAERS_BASE}{name}{FAERS_EXT}"        # base_url + quarter, exactly like src/ingest.py
                print("Downloading", url)
                try:
                    with requests.get(url, stream=True, headers={"User-Agent": "Mozilla/5.0"}, timeout=120) as r:
                        r.raise_for_status()
                        with open(zpath, "wb") as f:
                            for chunk in r.iter_content(1 << 20): f.write(chunk)
                    break
                except Exception as e:
                    last = e; print("  failed:", repr(e))
            else:
                raise RuntimeError(f"Could not download {q}. Download the ZIP by hand from the FDA FAERS page, upload it to {WORK}/ and re-run.") from last
        print("Unzipping", zpath.name)
        with zipfile.ZipFile(zpath) as z: z.extractall(outdir)
    return outdir

def find_table(outdir, kind):
    hits = [p for p in outdir.rglob("*") if p.suffix.lower() == ".txt" and p.name.lower().startswith(kind)]
    if not hits: raise FileNotFoundError(f"No {kind.upper()} file under {outdir}. Files: {[p.name for p in outdir.rglob('*')][:20]}")
    return hits[0]

paths = {k: [] for k in NEEDED}
for q in QUARTERS:
    out = fetch_quarter(q)
    for k in NEEDED:
        p = find_table(out, k); paths[k].append(p)
        print(f"{k.upper():5s} {p.name:20s} {p.stat().st_size/1e6:8.1f} MB")

## 3 · Loading functions — identical logic for CPU and GPU


In [ ]:
import numpy as np
import pandas as pd

def header_of(path):
    with open(path, encoding=ENCODING) as f:
        return f.readline().rstrip("\r\n").split(DELIMITER)

def read_pandas(path, kind):
    hdr = header_of(path)
    want = [h for h in hdr if h.lower() in NEEDED[kind]]
    df = pd.read_csv(path, sep=DELIMITER, usecols=want, dtype=str, quoting=3, encoding=ENCODING,
                     on_bad_lines="skip", low_memory=False)
    df.columns = [c.lower() for c in df.columns]
    return df

GPU_LOAD_MODE = {}
def read_cudf(path, kind):
    hdr = header_of(path)
    want = [h for h in hdr if h.lower() in NEEDED[kind]]
    try:
        df = cudf.read_csv(path, sep=DELIMITER, usecols=want, dtype={c: "str" for c in want}, quoting=3)
        GPU_LOAD_MODE[kind] = "cudf.read_csv (GPU parser)"
    except Exception as e:                                    # e.g. non-UTF8 bytes -> parse on CPU, move to GPU
        df = cudf.from_pandas(read_pandas(path, kind))
        GPU_LOAD_MODE[kind] = f"pandas parse + cudf.from_pandas ({type(e).__name__})"
    df.columns = [c.lower() for c in df.columns]
    return df

def load_all(reader):
    frames = {}
    for kind in NEEDED:
        parts = [reader(p, kind) for p in paths[kind]]
        frames[kind] = parts[0] if len(parts) == 1 else (cudf if reader is read_cudf else pd).concat(parts, ignore_index=True)
    return frames

## 4 · Benchmark harness


In [ ]:
def sync():
    if HAS_GPU:
        try:
            import cupy; cupy.cuda.Stream.null.synchronize()
        except Exception:
            pass

def bench(fn, repeats=REPEATS, warmup=False):
    if warmup:
        fn(); sync()
    times, out = [], None
    for _ in range(repeats):
        gc.collect(); sync()
        t0 = time.perf_counter(); out = fn(); sync()
        times.append(time.perf_counter() - t0)
    return statistics.median(times), out

def to_pd(x):
    return x.to_pandas() if hasattr(x, "to_pandas") else x

def label(i):
    # make labels identical on both libraries: 10.0 (pandas float index) and 10 (cuDF int index) -> "10"
    try:
        f = float(i)
        return str(int(f)) if f.is_integer() else str(f)
    except (TypeError, ValueError):
        return str(i)

def canon(x):
    # turn any result into plain python numbers so CPU and GPU answers can be compared
    x = to_pd(x)
    if isinstance(x, pd.DataFrame): return {str(i): canon(r) for i, r in x.round(6).iterrows()}
    if isinstance(x, pd.Series):   return {label(i): (None if pd.isna(v) else round(float(v), 6)) for i, v in x.items()}
    if isinstance(x, dict):        return {str(k): canon(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [canon(v) for v in x]
    return float(x)

def close(a, b, tol=1e-6):
    if isinstance(a, dict): return a.keys() == b.keys() and all(close(a[k], b[k], tol) for k in a)
    if isinstance(a, list): return len(a) == len(b) and all(close(x, y, tol) for x, y in zip(a, b))
    if a is None or b is None: return a is b
    return abs(a - b) <= tol * max(1.0, abs(a), abs(b))

## 5 · The workload — seven typical EDA operations


In [ ]:
def op_missing_scan(lib, F):
    # share of missing values for every column of DEMO and DRUG  (the 'missing-rate bar' of Act 1)
    return {"demo": F["demo"].isna().mean(), "drug": F["drug"].isna().mean()}

def glp1_mask(F):
    # same matching rule as src/faers_pipeline.py: keyword in prod_ai OR drugname, insulin combinations excluded
    text = (F["drug"]["prod_ai"].fillna("") + " " + F["drug"]["drugname"].fillna("")).str.upper()
    return text.str.contains(GLP1_REGEX, regex=True) & ~text.str.contains(EXCLUDE_REGEX, regex=True)

def op_string_flag(lib, F):
    # normalise drug names and flag GLP-1 products with a regex (string-heavy work)
    return int(glp1_mask(F).sum())

def op_groupby_counts(lib, F):
    # role code counts + the 20 most frequent drug names (counts only: ties may be ordered differently)
    roles = F["drug"].groupby("role_cod").size()
    top = F["drug"]["drugname"].str.upper().value_counts().head(20)
    return {"roles": roles.sort_index(), "top20_counts": list(to_pd(top).values.astype("float64"))}

def op_dedupe(lib, F):
    # keep the latest case version per caseid (class: duplicates must be found and dropped)
    d = F["demo"][["caseid", "caseversion"]].copy()
    d["cv"] = d["caseversion"].astype("float64")
    d = d.sort_values(["caseid", "cv"]).drop_duplicates("caseid", keep="last")
    return int(len(d))

def glp1_primary_suspects(lib, F):
    ps = F["drug"][(F["drug"]["role_cod"] == "PS") & glp1_mask(F)]
    return ps[["primaryid"]].drop_duplicates()

def op_join(lib, F):
    # inner join GLP-1 primary-suspect reports with DEMO, then count by sex
    ps = glp1_primary_suspects(lib, F)
    m = ps.merge(F["demo"][["primaryid", "sex", "occp_cod"]], on="primaryid", how="inner")
    return {"rows": int(len(m)), "by_sex": m.groupby("sex").size().sort_index()}

def op_serious_by_reporter(lib, F):
    # E6-style: % serious by reporter type for GLP-1 primary-suspect reports (join + left join + group-by)
    ps = glp1_primary_suspects(lib, F)
    m = ps.merge(F["demo"][["primaryid", "occp_cod"]], on="primaryid", how="inner")
    ser = F["outc"][F["outc"]["outc_cod"].isin(SERIOUS_CODES)][["primaryid"]].drop_duplicates()   # team definition of serious
    ser["serious"] = 1
    m = m.merge(ser, on="primaryid", how="left")
    m["serious"] = m["serious"].fillna(0)
    g = m.groupby("occp_cod")["serious"].agg(["mean", "count"])
    return g.sort_index()

def op_age_bins(lib, F):
    # numeric parsing of a text column + 10-year bins (the histogram of Act 3)
    # to_numeric(errors="coerce") treats odd strings (" 5", "1e2", "inf", "+5") differently in pandas and cuDF,
    # so both libraries use the SAME explicit rule: keep only plain numbers such as 55 or 55.5, then cast.
    s = F["demo"]["age"].str.strip()
    ok = s.str.match(r"^[0-9]{1,3}([.][0-9]+)?$").fillna(False).astype(bool)
    age = s[ok].astype("float64")
    age = age[age <= 120]
    return ((age // 10) * 10).astype("int64").value_counts().sort_index()

OPS = [("Missing-rate scan", op_missing_scan), ("String match (GLP-1 regex)", op_string_flag),
       ("Group-by / value counts", op_groupby_counts), ("De-duplicate cases", op_dedupe),
       ("Join with DEMO", op_join), ("% serious by reporter (E6)", op_serious_by_reporter),
       ("Age bins", op_age_bins)]

def run_workload(lib, F, warmup):
    res = {}
    for name, fn in OPS:
        t, out = bench(lambda: canon(fn(lib, F)), warmup=warmup)
        res[name] = (t, out)
    return res

## 6 · Run the CPU benchmark (pandas)

In [ ]:
t_load_cpu, F_cpu = bench(lambda: load_all(read_pandas), repeats=1)
n_rows = {k: len(v) for k, v in F_cpu.items()}
print(f"pandas load: {t_load_cpu:.1f}s   rows: {n_rows}")
cpu = run_workload(pd, F_cpu, warmup=False)
for k, (t, _) in cpu.items(): print(f"  {k:30s} {t:8.3f}s")

## 7 · Run the GPU benchmark (RAPIDS cuDF)

In [ ]:
gpu, t_load_gpu, F_gpu = {}, np.nan, None
if HAS_GPU:
    t_load_gpu, F_gpu = bench(lambda: load_all(read_cudf), repeats=1)
    print(f"cuDF load:   {t_load_gpu:.1f}s   how the files were parsed: {GPU_LOAD_MODE}")
    gpu = run_workload(cudf, F_gpu, warmup=True)
    for k, (t, _) in gpu.items(): print(f"  {k:30s} {t:8.3f}s")
else:
    print("Skipped: no GPU / cuDF. Switch the runtime to GPU and re-run from the top.")

## 8 · Results table + correctness check ("insight differences")
`same_answer` = the CPU and GPU returned identical numbers (tolerance 1e-6). Anything `False` must be investigated before trusting the GPU output.

In [ ]:
rows = [{"operation": "Load 3 FAERS tables", "pandas_s": t_load_cpu, "cudf_s": t_load_gpu, "same_answer": np.nan}]
for name, _ in OPS:
    tc, rc = cpu[name]
    tg, rg = gpu[name] if name in gpu else (np.nan, None)
    rows.append({"operation": name, "pandas_s": tc, "cudf_s": tg, "same_answer": close(rc, rg) if rg is not None else np.nan})
res = pd.DataFrame(rows)
res["speedup_x"] = res["pandas_s"] / res["cudf_s"]
total_cpu = res.loc[res.operation != "Load 3 FAERS tables", "pandas_s"].sum()
total_gpu = res.loc[res.operation != "Load 3 FAERS tables", "cudf_s"].sum()
display(res.round(3))
print(f"\nAnalysis steps only (no loading): pandas {total_cpu:.2f}s  vs  cuDF {total_gpu:.2f}s  ->  {total_cpu/total_gpu:.1f}x faster" if HAS_GPU else "")
if HAS_GPU and (res["same_answer"].dropna() == False).any():
    print("⚠️  Some operations returned DIFFERENT answers on CPU and GPU – investigate before using them.")
elif HAS_GPU:
    print("✅ All operations returned identical answers on CPU and GPU.")

In [ ]:
# How do we know the GPU results are correct?  Every operation returns a small plain result; CPU and GPU results are
# compared key by key (tolerance 1e-6). If any operation differs, this cell prints exactly WHAT differs, so it can be explained.
def explain_mismatch(name):
    a, b = cpu[name][1], gpu[name][1]
    def flat(x, prefix=""):
        if isinstance(x, dict):
            for k, v in x.items(): yield from flat(v, f"{prefix}{k}/")
        elif isinstance(x, list):
            for i, v in enumerate(x): yield from flat(v, f"{prefix}{i}/")
        else:
            yield prefix.rstrip("/"), x
    fa, fb = dict(flat(a)), dict(flat(b))
    only_cpu, only_gpu = sorted(set(fa) - set(fb)), sorted(set(fb) - set(fa))
    diff = [(k, fa[k], fb[k]) for k in fa.keys() & fb.keys() if not close(fa[k], fb[k])]
    print(f"--- {name}")
    if only_cpu: print("  keys only in pandas:", only_cpu[:10])
    if only_gpu: print("  keys only in cuDF:  ", only_gpu[:10])
    for k, x, y in diff[:10]: print(f"  {k}: pandas={x}  cuDF={y}")

if HAS_GPU:
    bad = list(res.loc[res["same_answer"] == False, "operation"])
    if not bad:
        print("✅ All", int(res["same_answer"].notna().sum()), "compared operations match exactly (tolerance 1e-6).")
    for name in bad:
        explain_mismatch(name)

## 9 · Charts

### G1a · Time per operation   —   G1b · Speed-up
**Questions:** *How long does each EDA step take on CPU vs GPU?* and *Which steps benefit most?*  A log axis is used because times differ by orders of magnitude.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "serif", "figure.dpi": 110, "savefig.dpi": 200,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True, "axes.titleweight": "bold", "axes.titlesize": 12})
RED, GREEN, BLUE, GREY = "#C1553B", "#3D7A5A", "#2B6E9E", "#8A8F98"

PROJECT_ROOT = REPO_DIR
FIG_DIR = PROJECT_ROOT / "figures"; FIG_DIR.mkdir(exist_ok=True)
REPORT_DIR = PROJECT_ROOT / "reports"; REPORT_DIR.mkdir(exist_ok=True)
SAVED = []
def savefig(name):
    p = FIG_DIR / f"{name}.png"; plt.savefig(p, bbox_inches="tight"); SAVED.append(p); print("saved ->", p)

if HAS_GPU:
    r = res.iloc[::-1].reset_index(drop=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.5, 1]})
    y = np.arange(len(r)); h = 0.38
    ax1.barh(y + h/2, r["pandas_s"], h, color=RED, label="pandas (CPU)")
    ax1.barh(y - h/2, r["cudf_s"], h, color=GREEN, label="cuDF (GPU)")
    for i, row in r.iterrows():
        ax1.text(row["pandas_s"] * 1.08, i + h/2, f"{row['pandas_s']:.2f}s", va="center", fontsize=8)
        ax1.text(row["cudf_s"] * 1.08, i - h/2, f"{row['cudf_s']:.2f}s", va="center", fontsize=8)
    ax1.set_yticks(y); ax1.set_yticklabels(r["operation"]); ax1.set_xscale("log")
    ax1.set_xlabel("seconds (log scale, median of runs)"); ax1.legend(frameon=False, loc="lower right")
    ax1.set_ylabel("EDA operation"); ax1.set_title("G1a · Time per EDA operation")

    s = r.sort_values("speedup_x")
    ax2.barh(s["operation"], s["speedup_x"], color=[GREEN if v >= 1 else RED for v in s["speedup_x"]])
    for i, v in enumerate(s["speedup_x"]): ax2.text(v * 1.05, i, f"{v:.1f}×", va="center", fontsize=9)
    ax2.axvline(1, ls="--", color=GREY); ax2.set_xscale("log"); ax2.set_xlabel("speed-up  (pandas time ÷ cuDF time, log)  — right of dashed line = GPU faster")
    ax2.set_ylabel("EDA operation"); ax2.set_title("G1b · Where does the GPU help?")
    plt.tight_layout(); savefig("G1ab_cpu_vs_gpu_time_and_speedup"); plt.show()
else:
    print("No GPU results to plot.")

### G1c · Scalability
**Question:** *Does the GPU advantage grow as the data grows?*  We stack copies of the three tables (with new IDs so joins stay correct) to make 1×, 2×, 4× … the data and time the **whole analysis pipeline** (the seven operations) on each library. If the lines diverge, the GPU scales better.


In [ ]:
def scaled(lib, F, k):
    if k == 1: return F
    out = {}
    for name, df in F.items():
        parts = []
        for i in range(k):
            c = df.copy()
            c["primaryid"] = c["primaryid"] + f"_{i}"
            if "caseid" in c.columns: c["caseid"] = c["caseid"] + f"_{i}"
            parts.append(c)
        out[name] = lib.concat(parts, ignore_index=True)
    return out

def total_time(lib, F, warmup):
    return sum(bench(lambda fn=fn: canon(fn(lib, F)), repeats=2, warmup=warmup)[0] for _, fn in OPS)

scal = []
for k in SCALES:
    Fc = scaled(pd, F_cpu, k)
    tc = total_time(pd, Fc, warmup=False)
    tg = np.nan
    if HAS_GPU:
        Fg = scaled(cudf, F_gpu, k)
        tg = total_time(cudf, Fg, warmup=True)
        del Fg
    scal.append({"scale": k, "drug_rows_millions": len(Fc["drug"]) / 1e6, "pandas_s": tc, "cudf_s": tg})
    del Fc; gc.collect()
    print(f"{k}x  drug rows {scal[-1]['drug_rows_millions']:.1f}M   pandas {tc:.1f}s   cuDF {tg:.1f}s")
scal = pd.DataFrame(scal); scal["speedup_x"] = scal["pandas_s"] / scal["cudf_s"]
display(scal.round(2))

In [ ]:
if HAS_GPU:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
    ax1.plot(scal["drug_rows_millions"], scal["pandas_s"], "o-", color=RED, lw=2, label="pandas (CPU)")
    ax1.plot(scal["drug_rows_millions"], scal["cudf_s"], "o-", color=GREEN, lw=2, label="cuDF (GPU)")
    for _, r_ in scal.iterrows():
        ax1.text(r_["drug_rows_millions"], r_["pandas_s"], f" {r_['pandas_s']:.1f}s", color=RED, va="bottom", fontsize=9)
        ax1.text(r_["drug_rows_millions"], r_["cudf_s"], f" {r_['cudf_s']:.1f}s", color=GREEN, va="top", fontsize=9)
    ax1.set_xlabel("rows in DRUG table (millions)"); ax1.set_ylabel("seconds for the 7-step pipeline")
    ax1.legend(frameon=False); ax1.set_title("G1c · Scalability: total pipeline time")
    ax2.bar([f"{k}×" for k in scal["scale"]], scal["speedup_x"], color=GREEN)
    for i, v in enumerate(scal["speedup_x"]): ax2.text(i, v * 1.02, f"{v:.1f}×", ha="center", fontsize=10)
    ax2.axhline(1, ls="--", color=GREY); ax2.set_ylabel("speed-up (pandas ÷ cuDF)"); ax2.set_xlabel("data size multiplier")
    ax2.set_title("Speed-up vs data size")
    plt.tight_layout(); savefig("G1c_scalability"); plt.show()
else:
    print("No GPU results to plot.")

## 10 · Zero-code-change option: `cudf.pandas`
The cell below writes a tiny pandas script and times it **with and without** the accelerator.

In [ ]:
script = Path(WORK) / "_zero_change_demo.py"
script.write_text(textwrap.dedent('''
    import sys, time, pandas as pd
    p = sys.argv[1]
    t = time.perf_counter()
    d = pd.read_csv(p, sep="__DELIM__", usecols=["primaryid", "role_cod", "drugname"], dtype=str, quoting=3, encoding="__ENC__", on_bad_lines="skip", low_memory=False)
    d["name"] = d["drugname"].fillna("").str.upper()
    flag = d["name"].str.contains("SEMAGLUTIDE|TIRZEPATIDE|LIRAGLUTIDE|DULAGLUTIDE")
    out = d[flag].groupby("role_cod").size()
    print(out.to_dict(), f"-> {time.perf_counter() - t:.1f}s")
''').replace('__DELIM__', DELIMITER).replace('__ENC__', ENCODING))
zc = {}
if HAS_GPU:
    for label, cmd in [("plain pandas", f"python {script} {paths['drug'][0]}"),
                       ("python -m cudf.pandas", f"python -m cudf.pandas {script} {paths['drug'][0]}")]:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print(f"{label:24s}", (r.stdout.strip() or r.stderr.strip()[-300:]))
else:
    print("Skipped (no GPU).")

## 11 · Insight, decision and caveats

In [ ]:
print("INSIGHT (auto-generated – speed numbers depend on the Colab GPU you were given):")
if HAS_GPU:
    best = res.dropna(subset=["speedup_x"]).iloc[1:].sort_values("speedup_x")
    print(f"  • Whole analysis pipeline (without loading): pandas {total_cpu:.1f}s vs cuDF {total_gpu:.1f}s = {total_cpu/total_gpu:.1f}× faster at 1× data.")
    print(f"  • Largest gain: '{best.iloc[-1]['operation']}' ({best.iloc[-1]['speedup_x']:.1f}×); smallest: '{best.iloc[0]['operation']}' ({best.iloc[0]['speedup_x']:.1f}×).")
    print(f"  • At {scal.iloc[-1]['scale']:.0f}× data the pipeline is {scal.iloc[-1]['speedup_x']:.1f}× faster on the GPU (at 1×: {scal.iloc[0]['speedup_x']:.1f}×).")
    print(f"  • CPU and GPU answers identical for all operations: {bool(res['same_answer'].dropna().all())}.")
else:
    print("  (run on a GPU runtime to generate this)")
print()
print("DECISION:")
print("  1) For the final 2022–2026 data (many quarters, so far more rows than this test), run joins, group-bys and string matching with cuDF if the speed-up above grows with data size.")
print("  2) Keep pandas for small summary tables and for plotting (matplotlib/seaborn need pandas objects: call .to_pandas()).")
print("  3) In the technical report quote the table + G1a/G1b/G1c as the explicit CPU-vs-GPU comparison the rubric asks for.")
print()
print("CAVEATS to state honestly in the report:")
print("  - One free Colab GPU (usually a T4) and a shared CPU: timings vary run to run; we report the median.")
print("  - The cuDF CSV reader may fall back to a CPU parse for non-UTF-8 text (see GPU_LOAD_MODE above); load speed-up is therefore reported separately from the analysis speed-up.")
print("  - Speed benchmark only: not the cleaned project dataset, so none of these counts are project results.")

## 12 · Save results

In [ ]:
res.round(4).to_csv(REPORT_DIR / "G1_cpu_vs_gpu_results.csv", index=False)
scal.round(4).to_csv(REPORT_DIR / "G1_scalability_results.csv", index=False)
print("saved tables to", REPORT_DIR)

if IN_COLAB:
    zpath = "/content/karishma_G1_outputs.zip"
    with zipfile.ZipFile(zpath, "w") as z:
        for p in SAVED: z.write(p, arcname=f"figures/{p.name}")
        for p in [REPORT_DIR / "G1_cpu_vs_gpu_results.csv", REPORT_DIR / "G1_scalability_results.csv"]:
            z.write(p, arcname=f"reports/{p.name}")
    from google.colab import files
    files.download(zpath)